# SMOTE for Images - Basic Pipeline Usage

This notebook demonstrates the basic usage of the SMOTE Image Synthesis pipeline.

In [ ]:
import torch
import numpy as np
import sys
import os

# Add the project root to Python path
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Import pipeline components
from smote_image_synthesis.data.models import PipelineConfig
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.smote.constrained_smote import ConstrainedSMOTE
from smote_image_synthesis.quality.assessor import QualityAssessor
from smote_image_synthesis.pipeline import SynthesisPipeline

## Create Configuration

Set up the pipeline configuration with appropriate parameters.

In [ ]:
# Create pipeline configuration
config = PipelineConfig(
    config_name="basic_example",
    output_dir="./basic_example_output"
)

# Adjust configuration for basic example
config.encoder_config.architecture = 'resnet18'
config.encoder_config.embedding_dim = 128
config.encoder_config.pretrained = False  # Use non-pretrained for faster execution
config.decoder_config.image_shape = (3, 64, 64)  # (channels, height, width)
config.decoder_config.decoder_type = 'autoencoder'
config.smote_config.k_neighbors = 3
config.quality_config.metrics = ['mse', 'mae']  # Use simple metrics for basic example

print("Configuration created:")
print(f"- Encoder: {config.encoder_config.architecture}")
print(f"- Embedding dimension: {config.encoder_config.embedding_dim}")
print(f"- Decoder type: {config.decoder_config.decoder_type}")
print(f"- Image shape: {config.decoder_config.image_shape}")
print(f"- SMOTE k_neighbors: {config.smote_config.k_neighbors}")

## Create Sample Data

Generate a small synthetic dataset to demonstrate the pipeline.

In [ ]:
# Create synthetic dataset
def create_synthetic_dataset(n_samples=100, image_size=64):
    """Create a synthetic dataset for demonstration."""
    images = []
    labels = []
    
    # Create imbalanced distribution
    class_sizes = [int(n_samples * 0.6), int(n_samples * 0.3), int(n_samples * 0.1)]
    
    # Class 0: Horizontal stripes (majority)
    for i in range(class_sizes[0]):
        img = torch.zeros(3, image_size, image_size)
        stripe_width = 8
        for y in range(0, image_size, stripe_width * 2):
            img[:, y:y+stripe_width, :] = torch.rand(3, 1, 1) * 0.8 + 0.2
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(0)
    
    # Class 1: Vertical stripes (minority)
    for i in range(class_sizes[1]):
        img = torch.zeros(3, image_size, image_size)
        stripe_width = 8
        for x in range(0, image_size, stripe_width * 2):
            img[:, :, x:x+stripe_width] = torch.rand(3, 1, 1) * 0.8 + 0.2
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(1)
    
    # Class 2: Checkerboard pattern (minority)
    for i in range(class_sizes[2]):
        img = torch.zeros(3, image_size, image_size)
        square_size = 16
        
        for y in range(0, image_size, square_size):
            for x in range(0, image_size, square_size):
                if (y // square_size + x // square_size) % 2 == 0:
                    color = torch.rand(3) * 0.8 + 0.2
                    img[:, y:y+square_size, x:x+square_size] = color.view(3, 1, 1)
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(2)
    
    images_tensor = torch.stack(images)
    labels_array = np.array(labels)
    
    print(f"Dataset created with shape {images_tensor.shape}")
    print(f"Class distribution: {np.bincount(labels_array)}")
    
    return images_tensor, labels_array

# Create dataset
images, labels = create_synthetic_dataset(n_samples=60, image_size=64)

## Set Up Pipeline Components

Initialize the encoder, decoder, SMOTE, and quality assessor.

In [ ]:
# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create encoder
encoder = ResNetEncoder(
    architecture=config.encoder_config.architecture,
    embedding_dim=config.encoder_config.embedding_dim,
    pretrained=config.encoder_config.pretrained,
    device=device
)

# Create decoder
decoder = AutoencoderDecoder(
    embedding_dim=config.encoder_config.embedding_dim,
    image_shape=config.decoder_config.image_shape,
    device=device
)

# Create SMOTE
smote = ConstrainedSMOTE(
    k_neighbors=config.smote_config.k_neighbors,
    use_clustering=config.smote_config.use_clustering,
    clustering_method=config.smote_config.cluster_method,
    random_state=config.seed
)

# Create quality assessor
quality_assessor = QualityAssessor(
    metrics=config.quality_config.metrics,
    device=device
)

print("Pipeline components created successfully!")

## Create and Fit Pipeline

Create the synthesis pipeline and fit it on the training data.

In [ ]:
# Create pipeline
pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

print("Fitting pipeline on training data...")
# Fit pipeline (this encodes images and fits SMOTE)
# For this basic example, we won't train the decoder separately
pipeline.fit(images, labels, train_decoder=False)

print("Pipeline fitted successfully!")

## Generate Synthetic Images

Generate synthetic images to balance the dataset.

In [ ]:
# Generate synthetic images
print("Generating synthetic images...")
n_synthetic = 30  # Generate 30 synthetic images
synthetic_images, synthetic_labels = pipeline.generate_synthetic_images(n_synthetic)

print(f"Generated {len(synthetic_images)} synthetic images")
print(f"Synthetic class distribution: {np.bincount(synthetic_labels) if len(synthetic_labels) > 0 else 'None'}")

# Show original vs synthetic class distribution
print(f"Original class distribution: {np.bincount(labels)}")

## Evaluate Quality

Evaluate the quality of the synthetic images.

In [ ]:
# Evaluate quality
if len(synthetic_images) > 0:
    print("Evaluating quality of synthetic images...")
    
    # Use a subset for evaluation to avoid memory issues
    n_eval = min(10, len(images), len(synthetic_images))
    eval_real = images[:n_eval]
    eval_synthetic = synthetic_images[:n_eval]
    
    quality_results = pipeline.evaluate_quality(eval_synthetic, eval_real)
    
    print("Quality Results:")
    for metric, value in quality_results.items():
        if isinstance(value, dict):
            for sub_metric, sub_value in value.items():
                print(f"  {metric}.{sub_metric}: {sub_value:.6f}")
        else:
            print(f"  {metric}: {value:.6f}")
else:
    print("No synthetic images generated to evaluate.")

## Summary

This notebook demonstrated the basic usage of the SMOTE Image Synthesis pipeline:

1. Created a configuration for the pipeline
2. Generated a synthetic dataset with class imbalance
3. Set up encoder, decoder, SMOTE, and quality assessment components
4. Fitted the pipeline on the training data
5. Generated synthetic images to balance the dataset
6. Evaluated the quality of the synthetic images

The pipeline successfully generated synthetic images that can be used to address class imbalance in image datasets.